# Radioactive decay

*nuclear · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/radioactive_decay)

Each nucleus decays at random with the same probability per second $\lambda$. Simulating a whole grid of nuclei as one boolean tensor — one coin toss per nucleus per time step — reproduces the smooth exponential law $N(t) = N_0 e^{-\lambda t}$ and the half-life $T_{1/2} = \ln 2/\lambda$.

**Vector form**

$$\frac{dN}{dt} = -\lambda N \;\;\Rightarrow\;\; N(t) = N_0\,e^{-\lambda t},\qquad T_{1/2} = \frac{\ln 2}{\lambda}$$

**Index (tensor) form**

$$s_n(t+\Delta t) = s_n(t)\,\big[u_n \ge p\big],\quad p = 1 - e^{-\lambda\Delta t},\qquad N(t) = \textstyle\sum_n s_n(t)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\lambda$ | `decay_constant(half_life)` | scalar | decay constant $\ln 2 / T_{1/2}$ |
| $s_n$ | `alive` | (S, S) | True while nucleus $n$ has not decayed; the grid is just a way to draw $n$ |
| $u_n$ | `torch.rand(alive.shape, generator=gen)` | (S, S) | one uniform random number per nucleus per step |
| $p$ | `decay_probability(lam, dt)` | scalar | chance to decay within one step $\Delta t$ |
| $s_n(t+\Delta t)$ | `alive & (u >= p)` | (S, S) | a decayed nucleus stays decayed |
| $\sum_n s_n$ | `alive.sum(dim=(-2, -1))` | (T,) | sum over all nuclei: the count $N(t)$ |

The exponential law is a statement about averages. With $N_0$ nuclei the count after time $t$ is binomial with mean $N_0 e^{-\lambda t}$ and standard deviation $\sqrt{N_0 e^{-\lambda t}(1 - e^{-\lambda t})}$, so a small grid wobbles visibly around the curve while a large grid follows it closely. After each half-life about half of the remaining nuclei are gone, however many there were. The step $\Delta t$ only needs to be small compared with $T_{1/2}$; the exact per-step probability $1 - e^{-\lambda\Delta t}$ makes the mean correct for any step. The random seed is fixed so the picture is repeatable.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/radioactive_decay/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import count, decay_constant, exact_count


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lam = decay_constant(params["half_life"])
    alive0 = tensors.nuclei(params["n_side"], dev)
    times = tensors.time_axis(params["t_max"], params["dt"], dev)

    history = solve.simulate(alive0, lam, params["dt"], len(times) - 1)  # (T, S, S)
    counts = count(history)  # (T,)
    exact = exact_count(alive0.numel(), lam, times)  # (T,)

    print(f"λ = {lam:.4f} 1/s   half-life = {params['half_life']:.2f} s   N0 = {alive0.numel()}")
    for k in range(0, len(times), max(1, len(times) // 8)):
        print(
            f"t = {times[k].item():6.2f} s   simulated N = {counts[k].item():5d}   N0 e^(-λt) = {exact[k].item():8.1f}"
        )
    return plot.animate(times, history, counts, exact, params["half_life"], out_dir / "radioactive_decay.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/radioactive_decay/equation.py
r"""Physics → tensor translation.

Every unstable nucleus has the same chance per unit time λ to decay, so the expected number left obeys

    dN/dt = -λ N    ⇒    N(t) = N_0 e^{-λ t},    half-life  T_{1/2} = \ln 2 / λ.

In a short step Δt one nucleus decays with probability p = 1 - e^{-λ Δt}. With s_n = 1 (still there) or 0
(decayed) for nucleus n and a uniform random number u_n ∈ [0, 1):

    s_n(t + Δt) = s_n(t) · [u_n ≥ p],        N(t) = s_n(t)   summed over n.
"""

import math

import torch


def decay_constant(half_life: float) -> float:
    r"""λ = \ln 2 / T_{1/2}."""
    return math.log(2.0) / half_life


def exact_count(n0: int, lam: float, t: torch.Tensor) -> torch.Tensor:
    r"""N(t) = N_0 e^{-λ t}.   t: (T,) → (T,)"""
    return n0 * torch.exp(-lam * t)


def decay_probability(lam: float, dt: float) -> float:
    r"""p = 1 - e^{-λ Δt}: chance that one nucleus decays during one step."""
    return 1.0 - math.exp(-lam * dt)


def decay_step(alive: torch.Tensor, u: torch.Tensor, p: float) -> torch.Tensor:
    r"""s_n(t + Δt) = s_n(t) · [u_n ≥ p].   alive, u: same shape (any) → bool tensor of that shape"""
    return alive & (u >= p)


def count(alive: torch.Tensor) -> torch.Tensor:
    r"""N = s_n summed over the nuclei (the last two axes form the grid).   (..., S, S) → (...,)"""
    return alive.sum(dim=(-2, -1))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/radioactive_decay/tensors.py
"""Build the concrete tensors: a square grid of nuclei and the time axis."""

import torch


def nuclei(n_side: int, device: torch.device) -> torch.Tensor:
    """s_n = True for every nucleus at t = 0, arranged as an (S, S) grid so we can draw it: shape (S, S) bool."""
    return torch.ones(n_side, n_side, dtype=torch.bool, device=device)


def time_axis(t_max: float, dt: float, device: torch.device) -> torch.Tensor:
    """t_k = k Δt for k = 0 … n_steps: shape (T,)."""
    n_steps = int(round(t_max / dt))
    return torch.arange(n_steps + 1, device=device, dtype=torch.float64) * dt


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/radioactive_decay/solve.py
"""Monte Carlo: one Bernoulli trial per nucleus per time step, all nuclei at once."""

import torch

from problems._common.progress import steps

from .equation import decay_probability, decay_step


def simulate(alive0: torch.Tensor, lam: float, dt: float, n_steps: int, seed: int = 0) -> torch.Tensor:
    """Return the state of every nucleus at every step: shape (T, *alive0.shape) bool.

    alive0 can carry extra leading axes (e.g. R repeated experiments of an S×S grid); each step draws one
    uniform number per nucleus and keeps it only if u ≥ p. A fixed seed makes the run repeatable.
    """
    gen = torch.Generator(device=alive0.device).manual_seed(seed)
    p = decay_probability(lam, dt)
    alive = alive0
    history = [alive]
    for _ in steps(n_steps, "decay steps"):
        u = torch.rand(alive.shape, generator=gen, device=alive.device)  # one random number per nucleus
        alive = decay_step(alive, u, p)
        history.append(alive)
    return torch.stack(history)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/radioactive_decay/plot.py
"""Animate the grid of nuclei decaying next to the counted N(t) and the exponential law."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.colors import ListedColormap

from problems._common.plotting import frame_indices, save_gif


def animate(
    times: torch.Tensor, history: torch.Tensor, counts: torch.Tensor, exact: torch.Tensor, half_life: float, path: Path
) -> Path:
    t = times.cpu().numpy()
    grid = history.cpu().numpy().astype(int)  # (T, S, S) 1 = undecayed
    n_sim = counts.cpu().numpy()
    n_exact = exact.cpu().numpy()
    n0 = n_exact[0]

    fig, (ax_grid, ax_n) = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1, 1.3]})
    cmap = ListedColormap(["#e5e7eb", "#2563eb"])  # decayed, undecayed
    image = ax_grid.imshow(grid[0], cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
    ax_grid.set_xticks([])
    ax_grid.set_yticks([])
    ax_grid.grid(False)
    title = ax_grid.set_title("")

    ax_n.plot(t, n_exact, color="#9ca3af", lw=2, ls="--", label=r"$N_0 e^{-\lambda t}$")
    (line,) = ax_n.plot([], [], color="#2563eb", lw=1.8, label="simulated nuclei")
    for k in range(1, int(t[-1] / half_life) + 1):
        ax_n.axvline(k * half_life, color="#f59e0b", lw=0.8, alpha=0.6)
        ax_n.text(k * half_life, n0 * 1.02, f"{k}T½", ha="center", fontsize=8, color="#b45309")
    ax_n.set_xlim(0, t[-1])
    ax_n.set_ylim(0, n0 * 1.1)
    ax_n.set_xlabel("t (s)")
    ax_n.set_ylabel("undecayed nuclei N")
    ax_n.legend(loc="upper right", bbox_to_anchor=(1.0, 0.9), fontsize=8)

    def update(step: int):
        image.set_data(grid[step])
        line.set_data(t[: step + 1], n_sim[: step + 1])
        title.set_text(f"t = {t[step]:.1f} s   N = {int(n_sim[step])} / {int(n0)}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_side': 30,
    'half_life': 2.0,
    't_max': 10.0,
    'dt': 0.05
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.radioactive_decay")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.radioactive_decay.main")
problem_dir = Path("problems/nuclear/radioactive_decay")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
